In [1]:
!pip install faker

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ------------------------------------ --- 1.8/2.0 MB 12.6 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 10.1 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from faker import Faker


fake = Faker()
np.random.seed(42)

def generate_biased_medisync_data(num_rows=5000):
    """
    Generates inventory data with store-specific demand biases
    to ensure the ML Redistribution model can learn patterns.
    """
    data = []
    categories = ['Antibiotics', 'Analgesics', 'Antipyretics', 'Antiseptics', 'Supplements']
    stores = ['Store_A', 'Store_B', 'Store_C', 'Store_D']
    
    # Store Specialties 
    specialties = {
        'Store_A': 'Antibiotics',
        'Store_B': 'Supplements',
        'Store_C': 'Analgesics',
        'Store_D': 'Antipyretics'
    }

    for _ in range(num_rows):
     
        medicine_name = fake.bothify(text='Med-####-??').upper()
        category = np.random.choice(categories)
        store_id = np.random.choice(stores)
    
        base_sales = np.random.randint(5, 40)
        
       
        if specialties.get(store_id) == category:
            avg_monthly_sales = base_sales * 5  
        else:
            avg_monthly_sales = base_sales    
            
        current_stock = np.random.randint(10, 300)
        unit_price = round(np.random.uniform(10.0, 500.0), 2)
        
       
        days_until_expiry = np.random.randint(15, 365)
        
       
        months_to_expiry = days_until_expiry / 30
        is_waste = 1 if current_stock > (avg_monthly_sales * months_to_expiry) else 0

        data.append({
            'medicine_name': medicine_name,
            'category': category,
            'current_stock': current_stock,
            'avg_monthly_sales': avg_monthly_sales,
            'unit_price': unit_price,
            'days_until_expiry': days_until_expiry,
            'store_id': store_id,
            'will_expire_unused': is_waste
        })

    return pd.DataFrame(data)


print("Generating biased dataset for Medisync...")
df = generate_biased_medisync_data(5000)


df.to_csv('medisync_training_data.csv', index=False)


print("-" * 30)
print("SUCCESS: medisync_training_data.csv created.")
print(f"Total Rows: {len(df)}")
print(f"Waste Instances: {df['will_expire_unused'].sum()}")
print("-" * 30)
print(df.head())

Generating biased dataset for Medisync...
------------------------------
SUCCESS: medisync_training_data.csv created.
Total Rows: 5000
Waste Instances: 2385
------------------------------
  medicine_name      category  current_stock  avg_monthly_sales  unit_price  \
0   MED-4448-EG   Antiseptics            116                 19      392.05   
1   MED-1854-PS    Analgesics             97                135      173.52   
2   MED-7472-MN  Antipyretics            201                  6      496.18   
3   MED-0583-ZH   Antiseptics            262                 26      221.65   
4   MED-5636-VH  Antipyretics            197                 32      189.52   

   days_until_expiry store_id  will_expire_unused  
0                 35  Store_A                   1  
1                166  Store_C                   0  
2                175  Store_B                   1  
3                 63  Store_B                   1  
4                204  Store_C                   0  


In [4]:
# expire before sold
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
import joblib


df = pd.read_csv('medisync_training_data.csv')


X = df[['current_stock', 'avg_monthly_sales', 'days_until_expiry', 'unit_price']]
y = df['will_expire_unused']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)


waste_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5))
])


print("Training Medisync Waste Prediction Model...")
waste_pipeline.fit(X_train, y_train)


y_pred = waste_pipeline.predict(X_test)
print("\n--- Model Performance Report ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print(classification_report(y_test, y_pred))


joblib.dump(waste_pipeline, 'medisync_waste_model.pkl')
print("Model saved as 'medisync_waste_model.pkl'")


def get_waste_prediction(stock, sales, days, price):
    """Returns probability of expiry before use."""
   
    features = pd.DataFrame([[stock, sales, days, price]], 
                           columns=['current_stock', 'avg_monthly_sales', 'days_until_expiry', 'unit_price'])
    
    prob = waste_pipeline.predict_proba(features)[0][1]
    return round(prob, 2)


print(f"\nSample Prediction (High Stock/Low Sales): {get_waste_prediction(100, 2, 15, 50.0)}")

Training Medisync Waste Prediction Model...

--- Model Performance Report ---
Accuracy: 98.00%
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       523
           1       0.98      0.98      0.98       477

    accuracy                           0.98      1000
   macro avg       0.98      0.98      0.98      1000
weighted avg       0.98      0.98      0.98      1000

Model saved as 'medisync_waste_model.pkl'

Sample Prediction (High Stock/Low Sales): 1.0
